# OBD Test-Set LLM-as-a-Judge Validation

This notebook evaluates only the OBD test-set outputs produced by the four experiment variants:

- Gemma + soft-prompt
- Gemma + Flamingo alignment
- Llama + soft-prompt
- Llama + Flamingo alignment

Protocol used here:

1. Load the four prediction files.
2. Merge them by test example `id`.
3. Judge each response independently with an LLM using an English rubric.
4. Aggregate criterion-level and overall scores.
5. Derive the best response per example from the weighted pointwise scores.

The notebook supports two judge backends:

- `openai`: GPT-style judge through API key
- `ollama`: local judge running through Ollama


In [10]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Any
from urllib import request

import pandas as pd


## Configuration

Set the judge backend and file paths below. The defaults point to the four OBD prediction files currently present in `data/`.


In [11]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = DATA_DIR / 'llm_judge_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_BACKEND = 'ollama'  # 'openai' or 'ollama'
OPENAI_MODEL = os.getenv('OPENAI_JUDGE_MODEL', 'gpt-4o-mini')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

OLLAMA_MODEL = os.getenv('OLLAMA_JUDGE_MODEL', 'qwen2.5:7b-instruct')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

FILE_MAP = {
    'gemma_soft_prompt': DATA_DIR / 'obd_eval_predictions_gemma3_270m.jsonl',
    'gemma_flamingo': DATA_DIR / 'obd_alignment_test_predictions_gemma3_270m.jsonl',
    'llama_soft_prompt': DATA_DIR / 'obd_eval_predictions_llama3_2_1B.jsonl',
    'llama_flamingo': DATA_DIR / 'obd_alignment_test_predictions_llama3_2_1B.jsonl',
}

SOURCE_DATASET = DATA_DIR / 'obd_cot_gpt5.jsonl'

MAX_CASES = None  # set an int for a small dry run
REQUEST_TIMEOUT = 180
SLEEP_BETWEEN_CALLS = 0.0
SAVE_EVERY = 10
FORCE_RERUN_POINTWISE = False
FORCE_RERUN_DIRECT_BEST_OF_FOUR = False

CRITERION_WEIGHTS = {
    'label_correctness': 0.30,
    'signal_grounding': 0.20,
    'reasoning_quality': 0.20,
    'coverage': 0.10,
    'format_compliance': 0.10,
    'text_quality': 0.10,
}

RUN_DIRECT_BEST_OF_FOUR = True

for name, path in FILE_MAP.items():
    print(f'{name}: {path} ->', 'FOUND' if path.exists() else 'MISSING')
print('Source dataset:', SOURCE_DATASET)
print('Output dir:', OUTPUT_DIR)
print('Force rerun pointwise:', FORCE_RERUN_POINTWISE)
print('Force rerun direct best-of-four:', FORCE_RERUN_DIRECT_BEST_OF_FOUR)


gemma_soft_prompt: c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\obd_eval_predictions_gemma3_270m.jsonl -> FOUND
gemma_flamingo: c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\obd_alignment_test_predictions_gemma3_270m.jsonl -> FOUND
llama_soft_prompt: c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\obd_eval_predictions_llama3_2_1B.jsonl -> FOUND
llama_flamingo: c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\obd_alignment_test_predictions_llama3_2_1B.jsonl -> FOUND
Source dataset: c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\obd_cot_gpt5.jsonl
Output dir: c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\llm_judge_outputs
Force rerun pointwise: False
Force rerun direct best-of-four: False


## Helpers

The source dataset is used to recover the original prompt for files that only contain `id`, `prediction`, and `target`.


In [12]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def extract_label(text: str | None) -> str | None:
    if not text:
        return None
    lowered = str(text).strip().lower()
    matches = re.findall(r'answer\s*:\s*([a-z_\-]+)', lowered)
    if matches:
        return matches[-1]
    for label in ('economical', 'normal', 'aggressive', 'congested'):
        if re.search(rf'\b{re.escape(label)}\b', lowered):
            return label
    return None


def compact_text(text: str | None) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def load_source_index(path: Path) -> dict[str, dict[str, Any]]:
    index = {}
    for row in read_jsonl(path):
        index[row['id']] = {
            'prompt': row.get('prompt') or row.get('pre_prompt') or '',
            'target': row.get('target') or row.get('answer') or '',
            'time_series_text': row.get('time_series_text') or [],
        }
    return index


def build_evaluation_rows(file_map: dict[str, Path], source_dataset: Path) -> list[dict[str, Any]]:
    source_index = load_source_index(source_dataset)
    rows_out = []
    for system_name, path in file_map.items():
        for row in read_jsonl(path):
            item_id = row['id']
            source = source_index.get(item_id, {})
            prompt = row.get('prompt') or source.get('prompt') or ''
            if not prompt and source.get('time_series_text'):
                prompt = ' | '.join([source.get('prompt', '')] + list(source.get('time_series_text', []))).strip(' |')
            target = row.get('target') or source.get('target') or ''
            rows_out.append({
                'id': item_id,
                'system_name': system_name,
                'prediction_path': str(path),
                'prompt': prompt,
                'target': target,
                'prediction': row.get('prediction', ''),
                'reference_label': extract_label(target),
                'predicted_label': extract_label(row.get('prediction', '')),
            })
    return rows_out


def validate_four_way_coverage(rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    counts = df.groupby('id')['system_name'].nunique().rename('system_count')
    print(counts.value_counts().sort_index().rename_axis('n_systems').to_frame('n_examples'))
    bad_ids = counts[counts != len(FILE_MAP)].index.tolist()
    if bad_ids:
        print(f'Warning: {len(bad_ids)} ids do not have all four systems.')
    else:
        print('All ids have the full four-way set of responses.')
    return df


def make_weighted_score(judgment: dict[str, Any]) -> float:
    return sum(float(judgment[key]) * weight for key, weight in CRITERION_WEIGHTS.items())


## Judge Prompt

The judge works in English and scores each response independently. Scores are criterion-level first; the winner can then be derived from the weighted pointwise scores.


In [13]:
JUDGE_SYSTEM_PROMPT = """You are a careful evaluator for vehicle telemetry reasoning tasks.
You must judge one candidate answer at a time.
Return valid JSON only.

Scoring rules:
- Score each criterion from 0 to 5 as an integer.
- Be strict about corrupted output, missing final labels, contradictions, and unsupported claims.
- Use the reference answer as the gold standard for both the final label and the expected reasoning content.
- Reward reasoning that is grounded in the telemetry description rather than generic driving-language filler.
- If the answer is unreadable, malformed, or mostly noise, assign very low scores.

Required JSON fields:
{
  \"label_correctness\": 0-5,
  \"signal_grounding\": 0-5,
  \"reasoning_quality\": 0-5,
  \"coverage\": 0-5,
  \"format_compliance\": 0-5,
  \"text_quality\": 0-5,
  \"overall_score\": 0-10,
  \"predicted_label\": \"string or null\",
  \"reference_label\": \"string or null\",
  \"label_match\": true or false,
  \"summary\": \"1-3 sentence explanation in English\"
}
"""


def build_pointwise_user_prompt(row: dict[str, Any]) -> str:
    return f"""Evaluate the following OBD response.

Task prompt:
{row['prompt']}

Reference answer:
{row['target']}

Candidate answer:
{row['prediction']}

Evaluation criteria:
1. label_correctness: Is the final class label correct relative to the reference?
2. signal_grounding: Does the explanation stay grounded in telemetry signal patterns such as speed, RPM, engine load, and throttle?
3. reasoning_quality: Does the conclusion follow logically from the described evidence?
4. coverage: Does the answer cover the main relevant signals and behavior patterns rather than only one narrow aspect?
5. format_compliance: Does it follow the expected response format and end with a valid 'Answer: <label>'?
6. text_quality: Is the text fluent, readable, and free from corruption, repetition, or obvious garbage output?

Return JSON only."""


def build_best_of_four_prompt(example_rows: list[dict[str, Any]]) -> str:
    base = example_rows[0]
    candidate_chunks = []
    for row in example_rows:
        candidate_chunks.append(
            f"System: {row['system_name']}\nCandidate answer:\n{row['prediction']}"
        )
    joined = '\n\n'.join(candidate_chunks)
    return f"""You are selecting the single best answer among four systems for one OBD example.
Return valid JSON only with this schema:
{{
  \"winner_system_name\": \"string\",
  \"reason\": \"short English explanation\"
}}

Task prompt:
{base['prompt']}

Reference answer:
{base['target']}

{joined}
"""


## Backend Clients

The code below uses simple HTTP calls so the notebook works with either OpenAI-compatible chat completions or a local Ollama server.


In [14]:
def post_json(url: str, payload: dict[str, Any], headers: dict[str, str] | None = None) -> dict[str, Any]:
    body = json.dumps(payload).encode('utf-8')
    req = request.Request(url, data=body, headers={**(headers or {}), 'Content-Type': 'application/json'})
    with request.urlopen(req, timeout=REQUEST_TIMEOUT) as response:
        return json.loads(response.read().decode('utf-8'))


def extract_json_block(text: str) -> dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def chat_with_openai(system_prompt: str, user_prompt: str) -> str:
    if not OPENAI_API_KEY:
        raise ValueError('OPENAI_API_KEY is not set.')
    payload = {
        'model': OPENAI_MODEL,
        'temperature': 0,
        'response_format': {'type': 'json_object'},
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    }
    headers = {'Authorization': f'Bearer {OPENAI_API_KEY}'}
    data = post_json('https://api.openai.com/v1/chat/completions', payload, headers=headers)
    return data['choices'][0]['message']['content']


def chat_with_ollama(system_prompt: str, user_prompt: str) -> str:
    payload = {
        'model': OLLAMA_MODEL,
        'stream': False,
        'format': 'json',
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        'options': {'temperature': 0},
    }
    data = post_json(f'{OLLAMA_BASE_URL}/api/chat', payload)
    return data['message']['content']


def judge_completion(system_prompt: str, user_prompt: str) -> dict[str, Any]:
    if JUDGE_BACKEND == 'openai':
        raw = chat_with_openai(system_prompt, user_prompt)
    elif JUDGE_BACKEND == 'ollama':
        raw = chat_with_ollama(system_prompt, user_prompt)
    else:
        raise ValueError(f'Unsupported JUDGE_BACKEND: {JUDGE_BACKEND}')
    return extract_json_block(raw)


def validate_judgment_schema(judgment: dict[str, Any]) -> dict[str, Any]:
    normalized = {}
    for key in CRITERION_WEIGHTS:
        normalized[key] = max(0, min(5, int(judgment[key])))
    normalized['overall_score'] = max(0.0, min(10.0, float(judgment['overall_score'])))
    normalized['predicted_label'] = judgment.get('predicted_label')
    normalized['reference_label'] = judgment.get('reference_label')
    normalized['label_match'] = bool(judgment.get('label_match', False))
    normalized['summary'] = compact_text(judgment.get('summary'))
    normalized['weighted_score'] = make_weighted_score(normalized)
    return normalized


## Load and Inspect the Four Test-Set Outputs


In [15]:
evaluation_rows = build_evaluation_rows(FILE_MAP, SOURCE_DATASET)
if MAX_CASES is not None:
    keep_ids = sorted({row['id'] for row in evaluation_rows})[:MAX_CASES]
    evaluation_rows = [row for row in evaluation_rows if row['id'] in keep_ids]

eval_df = validate_four_way_coverage(evaluation_rows)
print('Rows:', len(eval_df))
print('Examples:', eval_df['id'].nunique())
display(eval_df.head())


           n_examples
n_systems            
4                  39
All ids have the full four-way set of responses.
Rows: 156
Examples: 39


,id,system_name,prediction_path,prompt,target,prediction,reference_label,predicted_label
0,obd_cot_000088,gemma_soft_prompt,c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-Au...,You are shown a time-series plot of vehicle te...,Speed stays mostly low with gradual ramps and ...,coordinate coordinate 周期 TS_WINDOWUP 時間-序列 120...,normal,None
1,obd_cot_000177,gemma_soft_prompt,c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-Au...,You are shown a time-series plot of vehicle te...,Speed varies within a moderate band with smoot...,"context, time, velocity: 0.581, lat: 18.508, l...",normal,None
2,obd_cot_000041,gemma_soft_prompt,c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-Au...,You are shown a time-series plot of vehicle te...,The telemetry shows extended periods at very l...,:1540:000:000:000:000:000:000:000:000:000:000:...,economical,None
3,obd_cot_000164,gemma_soft_prompt,c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-Au...,You are shown a time-series plot of vehicle te...,"Across speed, RPM, engine load, and throttle, ...",26:59:36 -87:17:58 -86:55:51 -87:50:00 -86:50:...,normal,None
4,obd_cot_000089,gemma_soft_prompt,c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-Au...,You are shown a time-series plot of vehicle te...,Speed evolves mainly through gentle ramps with...,"=""-11.417847 -0.171633 -28.919177 <tex>$N=120;...",normal,None


## Run Pointwise LLM Judging

This is the main evaluation pass. Each of the four responses for each test example is judged independently.


In [ ]:
from tqdm.auto import tqdm

results_path = OUTPUT_DIR / f'obd_llm_judge_pointwise_{JUDGE_BACKEND}.jsonl'
summary_path = OUTPUT_DIR / f'obd_llm_judge_summary_{JUDGE_BACKEND}.json'

pointwise_results = []
completed_keys = set()
if results_path.exists() and not FORCE_RERUN_POINTWISE:
    pointwise_results = read_jsonl(results_path)
    completed_keys = {(row['id'], row['system_name']) for row in pointwise_results}
    print(f'Resuming from {len(pointwise_results)} saved judgments in {results_path.name}.')
elif results_path.exists() and FORCE_RERUN_POINTWISE:
    print(f'FORCE_RERUN_POINTWISE=True, ignoring existing file: {results_path.name}')

pending_rows = [row for row in evaluation_rows if (row['id'], row['system_name']) not in completed_keys]
print('Pending judgments:', len(pending_rows))
if not pending_rows:
    print('No pending judgments. Set FORCE_RERUN_POINTWISE = True to call the judge again from scratch.')

def save_partial_outputs(rows: list[dict[str, Any]]) -> None:
    write_jsonl(results_path, rows)
    partial_df = pd.DataFrame(rows)
    if partial_df.empty:
        return
    partial_summary = {
        'judge_backend': JUDGE_BACKEND,
        'judge_model': OPENAI_MODEL if JUDGE_BACKEND == 'openai' else OLLAMA_MODEL,
        'n_examples_scored': int(partial_df['id'].nunique()),
        'n_responses_scored': int(len(partial_df)),
        'systems_seen': sorted(partial_df['system_name'].unique().tolist()),
    }
    summary_path.write_text(json.dumps(partial_summary, indent=2, ensure_ascii=False), encoding='utf-8')

for idx, row in enumerate(tqdm(pending_rows, desc='Judging responses', unit='response'), start=1):
    judgment = validate_judgment_schema(
        judge_completion(JUDGE_SYSTEM_PROMPT, build_pointwise_user_prompt(row))
    )
    pointwise_results.append({**row, **judgment, 'judge_backend': JUDGE_BACKEND})

    save_partial_outputs(pointwise_results)

    if idx % SAVE_EVERY == 0 or idx == len(pending_rows):
        print(f'Saved {len(pointwise_results)} judgments to {results_path.name}')

    if SLEEP_BETWEEN_CALLS:
        time.sleep(SLEEP_BETWEEN_CALLS)

pointwise_df = pd.DataFrame(pointwise_results)
display(pointwise_df.head())


Pending judgments: 156


Judging responses:   0%|          | 0/156 [00:00<?, ?response/s]

Saved 10 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 20 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 30 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 40 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 50 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 60 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 70 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 80 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 90 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 100 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 110 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 120 judgments to obd_llm_judge_pointwise_ollama.jsonl
Saved 130 judgments to obd_llm_judge_pointwise_ollama.jsonl


## Aggregate Pointwise Results

The table below is the primary analysis for model-vs-approach comparison. A per-example winner is derived from the pointwise weighted score.


In [ ]:
criterion_cols = list(CRITERION_WEIGHTS) + ['overall_score', 'weighted_score']

system_summary = (
    pointwise_df.groupby('system_name')[criterion_cols]
    .mean()
    .sort_values(['weighted_score', 'overall_score'], ascending=False)
)
display(system_summary)

winner_rows = []
for item_id, group in pointwise_df.groupby('id'):
    ranked = group.sort_values(['weighted_score', 'overall_score'], ascending=False)
    top = ranked.iloc[0]
    runner_up = ranked.iloc[1] if len(ranked) > 1 else None
    winner_rows.append({
        'id': item_id,
        'winner_system_name': top['system_name'],
        'winner_weighted_score': float(top['weighted_score']),
        'winner_overall_score': float(top['overall_score']),
        'margin_vs_runner_up': float(top['weighted_score'] - runner_up['weighted_score']) if runner_up is not None else None,
    })

winner_df = pd.DataFrame(winner_rows)
display(winner_df.head())
display(winner_df['winner_system_name'].value_counts().rename_axis('system_name').to_frame('wins'))

summary_payload = {
    'judge_backend': JUDGE_BACKEND,
    'judge_model': OPENAI_MODEL if JUDGE_BACKEND == 'openai' else OLLAMA_MODEL,
    'n_examples': int(pointwise_df['id'].nunique()),
    'n_responses': int(len(pointwise_df)),
    'system_summary': system_summary.reset_index().to_dict(orient='records'),
    'winner_counts': winner_df['winner_system_name'].value_counts().to_dict(),
}
summary_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding='utf-8')
winner_df.to_csv(OUTPUT_DIR / f'obd_llm_judge_winners_{JUDGE_BACKEND}.csv', index=False)
system_summary.reset_index().to_csv(OUTPUT_DIR / f'obd_llm_judge_system_summary_{JUDGE_BACKEND}.csv', index=False)
print('Saved summary to', summary_path)


,label_correctness,signal_grounding,reasoning_quality,coverage,format_compliance,text_quality,overall_score,weighted_score
system_name,,,,,,,,
llama_soft_prompt,4.897436,4.871795,4.871795,4.871795,5.000000,4.897436,9.230769,4.894872
llama_flamingo,4.871795,4.820513,4.794872,4.794872,5.000000,4.923077,9.025641,4.856410
gemma_flamingo,4.871795,4.564103,4.564103,4.564103,5.000000,4.743590,7.384615,4.717949
gemma_soft_prompt,2.589744,2.717949,2.564103,2.692308,3.051282,2.692308,4.923077,2.676923


,id,winner_system_name,winner_weighted_score,winner_overall_score,margin_vs_runner_up
0,obd_cot_000003,llama_soft_prompt,5.0,10.0,0.0
1,obd_cot_000008,gemma_flamingo,5.0,10.0,0.0
2,obd_cot_000009,gemma_soft_prompt,5.0,10.0,0.0
3,obd_cot_000010,gemma_flamingo,5.0,10.0,0.0
4,obd_cot_000024,gemma_flamingo,5.0,10.0,0.0


,wins
system_name,
gemma_flamingo,18
llama_soft_prompt,12
gemma_soft_prompt,7
llama_flamingo,2


Saved summary to c:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\data\llm_judge_outputs\obd_llm_judge_summary_ollama.json


## Optional: Direct Best-of-Four Judge Pass

This is optional. The main protocol is still pointwise scoring. If enabled, this cell asks the judge to directly select the best of the four responses for each example.


In [ ]:
direct_choice_results = []

if RUN_DIRECT_BEST_OF_FOUR:
    direct_path = OUTPUT_DIR / f'obd_llm_judge_direct_best_{JUDGE_BACKEND}.jsonl'
    if direct_path.exists() and not FORCE_RERUN_DIRECT_BEST_OF_FOUR:
        direct_choice_results = read_jsonl(direct_path)
        print(f'Resuming direct best-of-four from {len(direct_choice_results)} saved judgments in {direct_path.name}.')
    elif direct_path.exists() and FORCE_RERUN_DIRECT_BEST_OF_FOUR:
        print(f'FORCE_RERUN_DIRECT_BEST_OF_FOUR=True, ignoring existing file: {direct_path.name}')
    completed_ids = {row['id'] for row in direct_choice_results}

    grouped = {item_id: group.to_dict(orient='records') for item_id, group in pointwise_df.groupby('id')}
    pending_ids = [item_id for item_id in grouped if item_id not in completed_ids]
    print('Pending direct best-of-four judgments:', len(pending_ids))
    if not pending_ids:
        print('No pending direct judgments. Set FORCE_RERUN_DIRECT_BEST_OF_FOUR = True to call the judge again from scratch.')

    for idx, item_id in enumerate(pending_ids, start=1):
        response = judge_completion('Return valid JSON only.', build_best_of_four_prompt(grouped[item_id]))
        direct_choice_results.append({
            'id': item_id,
            'winner_system_name': response['winner_system_name'],
            'reason': compact_text(response.get('reason')),
        })
        if idx % SAVE_EVERY == 0 or idx == len(pending_ids):
            write_jsonl(direct_path, direct_choice_results)
        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

    direct_choice_df = pd.DataFrame(direct_choice_results)
    display(direct_choice_df.head())
    display(direct_choice_df['winner_system_name'].value_counts().rename_axis('system_name').to_frame('direct_wins'))
else:
    print('RUN_DIRECT_BEST_OF_FOUR is False. Skipping this optional pass.')


Resuming direct best-of-four from 39 saved judgments in obd_llm_judge_direct_best_ollama.jsonl.
Pending direct best-of-four judgments: 0
No pending direct judgments. Set FORCE_RERUN_DIRECT_BEST_OF_FOUR = True to call the judge again from scratch.


,id,winner_system_name,reason
0,obd_cot_000003,llama_flamingo,The rationale provided is detailed and aligns ...
1,obd_cot_000008,llama_flamingo,The rationale provided is the most detailed an...
2,obd_cot_000009,llama_flamingo,The rationale provided is concise and directly...
3,obd_cot_000010,llama_flamingo,The answer provides a clear and concise ration...
4,obd_cot_000024,gemma_flamingo,Provides a clear and concise rationale that cl...


,direct_wins
system_name,
llama_flamingo,25
gemma_flamingo,10
gemma_soft_prompt,4


## Next Step

After you run the notebook, the key files will be saved under `data/llm_judge_outputs/`:

- pointwise raw judgments in JSONL
- system-level summary in CSV/JSON
- per-example winners in CSV

Recommended workflow:

1. Start with `MAX_CASES = 5` to validate the prompt and cost.
2. Run the full test set.
3. Compare the four systems primarily with the pointwise summary.
4. Use the optional direct best-of-four pass only if you want a second judge view of the winners.
